# E2: Episodic Trace Memory on ALFWorld

Validates the full episodic trace memory pipeline from the RFC spec:

1. **Ingest** expert trajectories as multi-step episodes
2. **Retrieve** analogous episodes by initial-state similarity + reward filter
3. **Inject** retrieved trajectories as procedural context
4. **Evaluate** task completion with and without episodic memory

**Dataset**: AgentInstruct ALFWorld split (336 GPT-4 expert trajectories, parsed from conversation format)

| Condition | Description |
|-----------|-------------|
| NoMemory | Zero-shot agent |
| RandomTrajectory | Random past trajectory injected |
| CVX-Episodic | Reward-filtered temporal analogy retrieval |

In [1]:
import os, json, time
import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# === MODEL CONFIG ===
# Option A: Ollama local (free, needs `ollama serve` running)
# Run on HPC: ollama pull qwen2.5-coder:7b-instruct
USE_OLLAMA = True
OLLAMA_MODEL = 'qwen2.5-coder:7b-instruct'
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'

# Option B: OpenAI API (set OPENAI_API_KEY env var)
# USE_OLLAMA = False
# OPENAI_MODEL = 'gpt-4o-mini'  # ~$0.10 for full run

if USE_OLLAMA:
    client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
    LLM_MODEL = OLLAMA_MODEL
else:
    assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY'
    client = OpenAI()
    LLM_MODEL = 'gpt-4o-mini'

EMBED_MODEL = 'all-MiniLM-L6-v2'
TOP_K = 3

DATA_DIR = Path('../data/episodic')
DATA_DIR.mkdir(parents=True, exist_ok=True)

embedder = SentenceTransformer(EMBED_MODEL)
D = embedder.get_sentence_embedding_dimension()
print(f'Embedding: {EMBED_MODEL} (D={D})')
print(f'LLM: {LLM_MODEL} ({"Ollama" if USE_OLLAMA else "OpenAI"})')

/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10114.09it/s]


BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding: all-MiniLM-L6-v2 (D=384)
LLM: qwen2.5-coder:7b-instruct (Ollama)


## 1. Load AgentInstruct Trajectories

Expert ALFWorld trajectories generated by GPT-4.

In [2]:
from datasets import load_dataset
import re

# AgentInstruct: expert ALFWorld trajectories (GPT-4)
raw_ds = load_dataset('zai-org/agentinstruct', split='alfworld')
print(f'AgentInstruct alfworld: {len(raw_ds)} trajectories')

# Parse conversation format into structured episodes
# These are GPT-4 expert trajectories — all are successful demonstrations (reward=1.0)
ds = []
for row in raw_ds:
    convs = row['conversations']
    
    # Extract task from the human turn that describes the environment
    task = ''
    steps = []
    for turn in convs:
        text = turn['value']
        # Task is in the turn containing "Your task is to:"
        if 'Your task is to:' in text:
            match = re.search(r'Your task is to:\s*(.+)', text)
            if match:
                task = match.group(1).strip()
        # GPT actions are the agent steps
        elif turn['from'] == 'gpt' and ('ACTION:' in text or 'THOUGHT:' in text):
            action_match = re.search(r'ACTION:\s*(.+)', text)
            action = action_match.group(1).strip() if action_match else ''
            thought_match = re.search(r'THOUGHT:\s*(.+?)(?:\n|ACTION:)', text, re.DOTALL)
            thought = thought_match.group(1).strip() if thought_match else ''
            if action:
                steps.append({'action': action, 'observation': thought})
        # Human feedback (environment response) — attach to last step
        elif turn['from'] == 'human' and len(steps) > 0 and 'Your task is to:' not in text:
            steps[-1]['observation'] = text[:200]
    
    if task and steps:
        # AgentInstruct trajectories are expert demonstrations — all successful
        reward = 1.0
        
        # Infer task type from task description
        task_lower = task.lower()
        if 'clean' in task_lower: task_type = 'clean'
        elif 'heat' in task_lower: task_type = 'heat'
        elif 'cool' in task_lower: task_type = 'cool'
        elif 'examine' in task_lower or 'look' in task_lower: task_type = 'examine'
        elif 'put' in task_lower or 'find' in task_lower: task_type = 'put'
        else: task_type = 'pick'
        
        ds.append({
            'task': task,
            'steps': steps,
            'reward': reward,
            'task_type': task_type,
        })

print(f'Parsed {len(ds)} episodes from AgentInstruct')
n_success = sum(1 for e in ds if e['reward'] >= 0.7)
print(f'  Successful: {n_success}/{len(ds)} (expert demonstrations)')
print(f'  Task types: {dict(pd.Series([e["task_type"] for e in ds]).value_counts())}')
print(f'  Steps/episode: min={min(len(e["steps"]) for e in ds)}, '
      f'median={np.median([len(e["steps"]) for e in ds]):.0f}, '
      f'max={max(len(e["steps"]) for e in ds)}')
print(f'\nSample task: {ds[0]["task"][:80]}')
print(f'Sample steps ({len(ds[0]["steps"])}):')
for s in ds[0]['steps'][:3]:
    print(f'  Action: {s["action"][:60]}')
    print(f'  Obs: {s["observation"][:60]}')

AgentInstruct alfworld: 336 trajectories
Parsed 336 episodes from AgentInstruct
  Successful: 336/336 (expert demonstrations)
  Task types: {'put': np.int64(158), 'clean': np.int64(68), 'cool': np.int64(49), 'examine': np.int64(37), 'heat': np.int64(24)}
  Steps/episode: min=3, median=10, max=35

Sample task: find two laptop and put them in bed.
Sample steps (14):
  Action: go to diningtable 1
  Obs: On the diningtable 1, you see a alarmclock 2, a bowl 2, a cd
  Action: take laptop 1 from diningtable 1
  Obs: You pick up the laptop 1 from the diningtable 1.
  Action: go to bed 1
  Obs: On the bed 1, you see a pillow 2, and a pillow 1.


## 2. Build Episodic Index

In [3]:
import chronos_vector as cvx

INDEX_PATH = str(DATA_DIR / 'alfworld_episodic.cvx')

def encode_entity(episode_id, step_index):
    return (episode_id << 16) | step_index

if os.path.exists(INDEX_PATH):
    index = cvx.TemporalIndex.load(INDEX_PATH)
    print(f'Loaded: {len(index)} points')
else:
    print('Building episodic memory...')
    index = cvx.TemporalIndex(m=16, ef_construction=100, ef_search=50)
    
    episode_data = {}
    
    for ep_idx, traj in enumerate(ds):
        task = traj['task'] if isinstance(traj, dict) else traj['task']
        steps = traj['steps'] if isinstance(traj, dict) else json.loads(traj['steps'])
        reward = traj['reward'] if isinstance(traj, dict) else float(traj.get('reward', 0))
        
        # Store each step as a temporal vector
        for s_idx, step in enumerate(steps):
            if isinstance(step, dict):
                text = f"Task: {task} | Action: {step.get('action', '')} | Obs: {step.get('observation', '')}"
            else:
                text = f"Task: {task} | Step: {step}"
            
            vec = embedder.encode(text[:500]).tolist()
            eid = encode_entity(ep_idx, s_idx)
            index.insert(entity_id=eid, timestamp=ep_idx * 10000 + s_idx, vector=vec)
        
        episode_data[ep_idx] = {
            'task': task,
            'steps': steps,
            'reward': reward,
            'n_steps': len(steps),
            'task_type': traj.get('task_type', 'unknown'),
        }
    
    index.save(INDEX_PATH)
    with open(str(DATA_DIR / 'alfworld_metadata.json'), 'w') as f:
        json.dump(episode_data, f)
    print(f'Built: {len(index)} points ({len(episode_data)} episodes)')

with open(str(DATA_DIR / 'alfworld_metadata.json')) as f:
    episode_data = {int(k): v for k, v in json.load(f).items()}

n_success = sum(1 for e in episode_data.values() if e['reward'] >= 0.7)
print(f'{len(episode_data)} episodes ({n_success} successful, {len(episode_data)-n_success} failed)')

Building episodic memory...


Built: 4542 points (336 episodes)
336 episodes (336 successful, 0 failed)


## 3. Retrieval & Evaluation

In [4]:
def retrieve_episodes(task_description, top_k=TOP_K, min_reward=0.7):
    """Retrieve successful episodes similar to the given task."""
    query_vec = embedder.encode(f'Task: {task_description}').tolist()
    results = index.search(vector=query_vec, k=top_k * 20)
    
    seen = set()
    episodes = []
    for node_id, ts, score in results:
        ep_idx = node_id // max(1, max(e['n_steps'] for e in episode_data.values()))
        # More robust: use timestamp to find episode
        ep_idx = ts // 10000
        
        if ep_idx in seen or ep_idx not in episode_data:
            continue
        
        ep = episode_data[ep_idx]
        if ep['reward'] < min_reward:  # reward filter!
            continue
        
        seen.add(ep_idx)
        episodes.append({'episode_id': ep_idx, 'similarity': score, **ep})
        
        if len(episodes) >= top_k:
            break
    
    return episodes


def format_trajectory(episodes):
    """Format episodes for LLM prompt."""
    lines = []
    for i, ep in enumerate(episodes, 1):
        outcome = 'Success' if ep['reward'] >= 0.7 else 'Failure'
        lines.append(f'[Past Episode {i} — {outcome}, {ep["n_steps"]} steps]')
        lines.append(f'Task: {ep["task"]}')
        for j, step in enumerate(ep['steps'][:10], 1):  # cap at 10 steps
            if isinstance(step, dict):
                lines.append(f'  Step {j}: {step.get("action", "")} → {step.get("observation", "")[:80]}')
            else:
                lines.append(f'  Step {j}: {str(step)[:100]}')
        lines.append('')
    return '\n'.join(lines)


# Test retrieval
test_task = episode_data[0]['task']
retrieved = retrieve_episodes(test_task)
print(f'Query: {test_task}')
print(f'Retrieved {len(retrieved)} episodes:')
for ep in retrieved:
    print(f'  [{ep["episode_id"]}] sim={ep["similarity"]:.3f}, reward={ep["reward"]}, steps={ep["n_steps"]}: {ep["task"][:50]}')

Query: find two laptop and put them in bed.
Retrieved 3 episodes:
  [0] sim=0.240, reward=1.0, steps=14: find two laptop and put them in bed.
  [259] sim=0.307, reward=1.0, steps=8: put two laptop in bed.
  [45] sim=0.493, reward=1.0, steps=17: find two book and put them in bed.


In [5]:
def solve_with_llm(task, context=''):
    """Ask LLM to produce a plan for the task."""
    system = (
        'You are an embodied agent in a household. Given a task, produce a step-by-step '
        'action plan. Each step should be one action like "go to kitchen", "take apple", '
        '"put apple in fridge". Output ONLY the numbered steps, no explanation.'
    )
    user_parts = []
    if context:
        user_parts.append('Here are examples of how similar tasks were solved:\n')
        user_parts.append(context)
        user_parts.append('\n---\nNow solve this task:\n')
    user_parts.append(f'Task: {task}')
    
    response = client.chat.completions.create(
        model=LLM_MODEL,
        messages=[
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': '\n'.join(user_parts)},
        ],
        temperature=0.0,
        max_tokens=512,
    )
    plan = response.choices[0].message.content
    n_steps = len([l for l in plan.strip().split('\n') if l.strip()])
    return plan, n_steps


# Evaluate: compare plan quality across conditions
# Without a real ALFWorld env, we measure:
# 1. Plan specificity (number of concrete action steps)
# 2. Similarity of plan to expert trajectory
# 3. Token efficiency

N_EVAL = min(30, len(episode_data))
eval_episodes = list(episode_data.items())[:N_EVAL]

metrics = {'NoMemory': [], 'RandomTrajectory': [], 'CVX-Episodic': []}

for ep_idx, ep in eval_episodes:
    task = ep['task']
    expert_steps = ep['n_steps']
    
    # NoMemory
    plan, n_steps = solve_with_llm(task)
    metrics['NoMemory'].append({'steps': n_steps, 'expert_steps': expert_steps})
    
    # RandomTrajectory
    random_eps = [episode_data[i] for i in np.random.choice(list(episode_data.keys()), 3)]
    plan, n_steps = solve_with_llm(task, format_trajectory(random_eps))
    metrics['RandomTrajectory'].append({'steps': n_steps, 'expert_steps': expert_steps})
    
    # CVX-Episodic (exclude the current episode)
    retrieved = retrieve_episodes(task, min_reward=0.7)
    retrieved = [e for e in retrieved if e['episode_id'] != ep_idx][:3]
    if retrieved:
        plan, n_steps = solve_with_llm(task, format_trajectory(retrieved))
    else:
        plan, n_steps = solve_with_llm(task)
    metrics['CVX-Episodic'].append({'steps': n_steps, 'expert_steps': expert_steps})
    
    if (ep_idx + 1) % 10 == 0:
        print(f'[{ep_idx+1}/{N_EVAL}] done')

print('\n=== RESULTS ===')
for cond, data in metrics.items():
    mean_steps = np.mean([d['steps'] for d in data])
    mean_expert = np.mean([d['expert_steps'] for d in data])
    step_ratio = mean_steps / max(mean_expert, 1)
    print(f'{cond:20s}: mean steps={mean_steps:.1f} (expert={mean_expert:.1f}, ratio={step_ratio:.2f})')

[10/30] done


[20/30] done


[30/30] done

=== RESULTS ===
NoMemory            : mean steps=6.4 (expert=14.0, ratio=0.45)
RandomTrajectory    : mean steps=6.6 (expert=14.0, ratio=0.47)
CVX-Episodic        : mean steps=8.1 (expert=14.0, ratio=0.58)


In [6]:
import plotly.graph_objects as go

cond_names = list(metrics.keys())
mean_steps = [np.mean([d['steps'] for d in metrics[c]]) for c in cond_names]

fig = go.Figure(data=go.Bar(
    x=cond_names, y=mean_steps,
    text=[f'{s:.1f}' for s in mean_steps],
    textposition='outside',
    marker_color=['#95a5a6', '#3498db', '#e74c3c'],
))
fig.update_layout(
    title=f'Mean Plan Steps by Condition (n={N_EVAL})',
    yaxis_title='Mean Steps', height=400,
)
fig.show()

## Summary

In [7]:
print('=== E2: EPISODIC ALFWORLD BENCHMARK ===')
print(f'Model: {LLM_MODEL}')
print(f'Episodes: {len(episode_data)} ({n_success} successful)')
print(f'Evaluated: {N_EVAL} tasks')
print()
for cond, data in metrics.items():
    mean_s = np.mean([d['steps'] for d in data])
    print(f'  {cond:20s}: {mean_s:.1f} mean steps')
print()
print('CVX features used: insert, search, episode encoding, reward filtering')
print('Note: Without real ALFWorld env, we compare plan length vs expert.')
print('Full evaluation requires alfworld package + environment execution.')

=== E2: EPISODIC ALFWORLD BENCHMARK ===
Model: qwen2.5-coder:7b-instruct
Episodes: 336 (336 successful)
Evaluated: 30 tasks

  NoMemory            : 6.4 mean steps
  RandomTrajectory    : 6.6 mean steps
  CVX-Episodic        : 8.1 mean steps

CVX features used: insert, search, episode encoding, reward filtering
Note: Without real ALFWorld env, we compare plan length vs expert.
Full evaluation requires alfworld package + environment execution.
